### 1. Carga de datos y filtro inicial

In [9]:
import pandas as pd
import numpy as np

recs = pd.read_csv("recommendations.csv")

# eliminar usuarios sin horas significativas jugadas
recs = recs[recs["hours"] > 0.1]

print("Original:")
print("users:", recs["user_id"].nunique(), "items:", recs["app_id"].nunique())


Original:
users: 13732369 items: 37509


La idea es quedarse solo con usuarios e ítems con suficiente actividad global.

### 2. Filtro global ex-ante (5/5)

In [10]:
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

user_counts = recs["user_id"].value_counts()
item_counts = recs["app_id"].value_counts()

good_users = user_counts[user_counts >= MIN_USER_INTERACTIONS].index
good_items = item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index

recs_filtered = recs[
    recs["user_id"].isin(good_users) &
    recs["app_id"].isin(good_items)
].copy()

print("\ndespues filtro global (MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS):")
print("users:", recs_filtered["user_id"].nunique(), "items:", recs_filtered["app_id"].nunique())




despues filtro global (MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS):
users: 1894391 items: 34916


Al igual que antes, se filtran usuarios e ítems con suficiente actividad.

### 3. Muestreo estratificado de usuarios

In [11]:
# 3. muestreo estratificado de usuarios

# actividad de usuario en el dataset filtrado
user_activity = recs_filtered["user_id"].value_counts().rename("n_interactions").reset_index()
user_activity.columns = ["user_id", "n_interactions"]

# buckets de actividad (low / mid / high)
user_activity["activity_bucket"] = pd.qcut(
    user_activity["n_interactions"],
    q=3,  # terciles
    labels=["low", "mid", "high"]
)

TARGET_USERS = 10000
bucket_weights = {
    "low": 0.4,
    "mid": 0.4,
    "high": 0.2,
}

bucket_users = {
    b: user_activity[user_activity["activity_bucket"] == b]["user_id"]
    for b in ["low", "mid", "high"]
}
bucket_counts = {b: len(u) for b, u in bucket_users.items()}
print("\nUsuarios por bucket (despues filtro global):", bucket_counts)

# 1) asignación inicial
alloc = {}
for b, w in bucket_weights.items():
    desired = int(TARGET_USERS * w)
    available = bucket_counts[b]
    alloc[b] = min(desired, available)

# 2) redistribuir cupos sobrantes
total_alloc = sum(alloc.values())
total_available = sum(bucket_counts.values())

if total_available <= TARGET_USERS:
    final_alloc = bucket_counts.copy()
else:
    leftover = TARGET_USERS - total_alloc
    spare = {b: bucket_counts[b] - alloc[b] for b in bucket_weights.keys()}
    buckets_by_spare = sorted(spare.keys(), key=lambda b: spare[b], reverse=True)
    final_alloc = alloc.copy()
    for b in buckets_by_spare:
        if leftover <= 0:
            break
        can_take = spare[b]
        if can_take <= 0:
            continue
        take = min(can_take, leftover)
        final_alloc[b] += take
        leftover -= take

print("asignacion final de usuarios por bucket:", final_alloc)
print("total usuarios muestreados:", sum(final_alloc.values()))

# 3) samplear usuarios por bucket
sampled_users_list = []
for b, size in final_alloc.items():
    if size <= 0:
        continue
    sampled_users_list.append(bucket_users[b].sample(size, random_state=42))

sampled_users = pd.concat(sampled_users_list)
print("usuarios unicos muestreados:", sampled_users.nunique())




Usuarios por bucket (despues filtro global): {'low': 730112, 'mid': 586700, 'high': 577579}
asignacion final de usuarios por bucket: {'low': 4000, 'mid': 4000, 'high': 2000}
total usuarios muestreados: 10000
usuarios unicos muestreados: 10000


La idea en el fondo es hacer un muestreo estratificado exacto a 10.000 usuarios en los tres niveles de actividad (40/40/20). Cosa que se logre tener representación de usuarios con poca, media y mucha actividad.

In [12]:
# 4. quedarnos con las interacciones de esos usuarios
# y filtrar items dentro del sample

sample_recs = recs_filtered[recs_filtered["user_id"].isin(sampled_users)].copy()

print("\ndespues de muestrear usuarios (antes filtro de items en sample):")
print("users:", sample_recs["user_id"].nunique(), "items:", sample_recs["app_id"].nunique())

MIN_ITEM_INTERACTIONS_SAMPLE = 5
item_counts_sample = sample_recs["app_id"].value_counts()
good_items_sample = item_counts_sample[item_counts_sample >= MIN_ITEM_INTERACTIONS_SAMPLE].index

sample_recs = sample_recs[sample_recs["app_id"].isin(good_items_sample)].copy()

print("despues del filtro de items dentro del sample:")
print("users:", sample_recs["user_id"].nunique(), "items:", sample_recs["app_id"].nunique())


despues de muestrear usuarios (antes filtro de items en sample):
users: 10000 items: 10122
despues del filtro de items dentro del sample:
users: 9991 items: 2880


Ahora, se limpia items con muy pocas interacciones dentro del sample.

### 5. Orden temporal + filtro usuarios con >=3 interacciones

In [13]:
sample_recs["date"] = pd.to_datetime(sample_recs["date"])
sample_recs = sample_recs.sort_values(["user_id", "date"])

# guardar el sample por separado, por si acaso
sample_recs.to_csv("data/sampled_recommendations.csv", index=False)

# filtrar usuarios con al menos 3 interacciones en el sample
user_counts_sample2 = sample_recs["user_id"].value_counts()
good_users_3plus = user_counts_sample2[user_counts_sample2 >= 3].index

df = sample_recs[sample_recs["user_id"].isin(good_users_3plus)].copy()
df = df.sort_values(["user_id", "date"])

print("\ndespues filtro usuarios con al menos 3 interacciones en el sample:")
print("users:", df["user_id"].nunique(), "items:", df["app_id"].nunique())




despues filtro usuarios con al menos 3 interacciones en el sample:
users: 9906 items: 2880


La idea es quedarse solo con usuarios que tienen historial mínimo para poder hacer train / val / test (al menos 3 puntos en el tiempo).

### 6. Split aprox. 80/10/10 temporal por usuario

In [14]:
train_list = []
val_list   = []
test_list  = []

for user, group in df.groupby("user_id"):
    n = len(group)

    # n >= 3 por construcción
    if n == 3:
        # 1 / 1 / 1
        train_u = group.iloc[:1]
        val_u   = group.iloc[1:2]
        test_u  = group.iloc[2:]
    elif n == 4:
        # 2 / 1 / 1
        train_u = group.iloc[:2]
        val_u   = group.iloc[2:3]
        test_u  = group.iloc[3:]
    elif n == 5:
        # 3 / 1 / 1
        train_u = group.iloc[:3]
        val_u   = group.iloc[3:4]
        test_u  = group.iloc[4:]
    else:
        # n >= 6 -> 80/10/10 aproximado
        idx_train = int(np.floor(0.8 * n))
        idx_val   = int(np.floor(0.9 * n))

        # asegurar que haya al menos 1 en val y 1 en test
        if idx_val <= idx_train:
            idx_val = idx_train + 1
        if idx_val >= n:
            idx_val = n - 1

        train_u = group.iloc[:idx_train]
        val_u   = group.iloc[idx_train:idx_val]
        test_u  = group.iloc[idx_val:]

    train_list.append(train_u)
    val_list.append(val_u)
    test_list.append(test_u)

train = pd.concat(train_list)
val   = pd.concat(val_list)
test  = pd.concat(test_list)

print("\nsplits finales:")
print("train:", train.shape, "users:", train["user_id"].nunique())
print("val:  ", val.shape,   "users:", val["user_id"].nunique())
print("test: ", test.shape,  "users:", test["user_id"].nunique())

total_interactions = len(df)
print(f"%train: {len(train)/total_interactions:.2%}")
print(f"%val:   {len(val)/total_interactions:.2%}")
print(f"%test:  {len(test)/total_interactions:.2%}")





splits finales:
train: (63905, 8) users: 9906
val:   (11649, 8) users: 9906
test:  (12784, 8) users: 9906
%train: 72.34%
%val:   13.19%
%test:  14.47%


Si bien no se puede garantizar un split exacto 80/10/10 para todos los usuarios, se busca aproximar lo más posible a eso, lo que se logra.

### 7. Guardar splits

In [15]:

train.to_csv("data/split/train_split.csv", index=False)
val.to_csv("data/split/val_split.csv", index=False)
test.to_csv("data/split/test_split.csv", index=False)